# Lab 1A · Images as arrays

**Computer Vision for Developers · SDAIA Academy** · Day 1 · Block 3 · Acts 2–3

**Goal:** Read an image as an array (shape, dtype, channels), fix the BGR bug, crop by slicing, threshold with a histogram, and write convolution by hand — then check it against OpenCV.

## How to use this notebook

1. A GPU is not needed for this lab; the CPU runtime is fine.
2. Run the cells **from top to bottom** with *Shift + Enter*. Do not skip cells: later cells use variables from earlier ones.
3. Cells with a **TODO** comment need **one line** from you. The cell right after each TODO is a check: it prints ✅ when your line is right, or a hint when it is not.
4. Nobody writes a cell from empty. If you are stuck for more than five minutes, open the *solved* version of this notebook and compare.
5. Colab forgets files when the session ends. If you come back later, run everything again from the top.
6. The **Extra tasks** at the end are optional and are not graded.

Expected time: about 45 minutes in class; finish the rest at home.

In [ ]:
# GPU check, with a CPU fallback
import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then Runtime → Restart.")
    print("   You can keep going on CPU: the notebook switches to the same settings (this lab is light).")
# FAST mode shrinks data and epochs so everything still finishes on a CPU.
FAST = DEVICE == "cpu" or os.environ.get("CVD_FAST") == "1"
print("DEVICE =", DEVICE, "| FAST =", FAST)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data

def show(*imgs, titles=None, cmap=None, size=4):
    """Show images side by side."""
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size))
    axes = np.atleast_1d(axes)
    for k, (ax, im) in enumerate(zip(axes, imgs)):
        ax.imshow(im, cmap=cmap if im.ndim == 2 else None, vmin=0 if im.ndim == 2 and im.dtype == np.uint8 else None,
                  vmax=255 if im.ndim == 2 and im.dtype == np.uint8 else None)
        ax.set_title(titles[k] if titles else "")
        ax.axis("off")
    plt.show()

print("NumPy", np.__version__, "| OpenCV", cv2.__version__)

## 2 · Images are just numbers

An image is a grid of numbers. A grayscale image is a 2D table; a colour image stacks three tables (red, green, blue). The first thing you check about any image is its **shape** and its **dtype**.

In [ ]:
astronaut = data.astronaut()   # a colour photo, already RGB
camera = data.camera()         # a grayscale photo
print("astronaut:", astronaut.shape, astronaut.dtype, "min", astronaut.min(), "max", astronaut.max())
print("camera:   ", camera.shape, camera.dtype)
print("top-left pixel of astronaut (R, G, B):", astronaut[0, 0])
show(astronaut, camera, titles=["astronaut (H, W, 3)", "camera (H, W)"], cmap="gray")

### 2.1 · Coordinates: `img[y, x]`, origin top-left

Rows come first (y grows **downward**), then columns (x grows to the right). Let us read one pixel and paint a small square so you can see where it is.

In [ ]:
marked = astronaut.copy()
y, x = 100, 300
print("pixel at row y=100, column x=300:", astronaut[y, x])
marked[y-10:y+10, x-10:x+10] = [255, 0, 0]      # paint a red square
show(marked, titles=["red square at y=100, x=300"])

### 2.2 · Gotcha: BGR vs RGB

OpenCV loads images in **B, G, R** order. Matplotlib, PIL and PyTorch expect **R, G, B**. Below we save the photo and read it back with OpenCV, exactly as you would with your own files.

In [ ]:
cv2.imwrite("astronaut.png", cv2.cvtColor(astronaut, cv2.COLOR_RGB2BGR))   # save like any photo on disk
bgr = cv2.imread("astronaut.png")                                           # OpenCV gives B, G, R
show(bgr, titles=["read with cv2.imread — colours look wrong"])

In [ ]:
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
show(rgb, titles=["fixed"]) if rgb is not None else print("Fill in the TODO first")

In [ ]:
# Check
try:
    assert np.array_equal(rgb, astronaut)
    print('✅ Colours fixed: rgb matches the original exactly.')
except Exception as e:
    print('❌ Not yet. Hint: use cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).')

### 2.3 · Cropping is just slicing

`img[y1:y2, x1:x2]` keeps rows y1 to y2 and columns x1 to x2. No function needed.

In [ ]:
face = rgb[30:180, 150:300]
show(face, titles=["face crop"]) if face is not None else print("Fill in the TODO first")

In [ ]:
# Check
try:
    assert face.shape == (150, 150, 3)
    print('✅ Crop has shape (150, 150, 3).')
except Exception as e:
    print('❌ Not yet. Hint: rows first: rgb[30:180, 150:300].')

### 2.4 · Gotcha: uint8 overflow

A uint8 holds 0–255. What is 200 + 100? Predict it before you run the cell.

In [ ]:
a = np.array([200], dtype=np.uint8)
print("uint8: 200 + 100 =", (a + 100)[0], " ← wrapped around (300 - 256)")
safe = np.clip(a.astype(np.int16) + 100, 0, 255).astype(np.uint8)
print("fixed: convert, add, clip →", safe[0])

brighter_wrong = astronaut + 100
brighter_right = np.clip(astronaut.astype(np.int16) + 100, 0, 255).astype(np.uint8)
show(brighter_wrong, brighter_right, titles=["rgb + 100 (overflow)", "convert, add, clip"])

### 2.5 · Histogram and thresholding

A histogram counts how many pixels have each value. Thresholding turns a grayscale image into black and white: above T → 255, below → 0.

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(camera.ravel(), bins=256, range=(0, 255), color="gray")
plt.title("histogram of camera"); plt.xlabel("pixel value"); plt.ylabel("count")
plt.show()

In [ ]:
T = 100
binary = np.where(camera > T, 255, 0).astype(np.uint8)
otsu_T, otsu = cv2.threshold(camera, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu picked T =", otsu_T)
show(binary if binary is not None else camera, otsu, titles=[f"global T={T}", f"Otsu T={otsu_T:.0f}"], cmap="gray")

In [ ]:
# Check
try:
    assert binary.dtype == np.uint8 and set(np.unique(binary)) <= {0, 255} and abs((binary == 255).mean() - (camera > 100).mean()) < 1e-9
    print('✅ Threshold is correct: only 0 and 255, uint8.')
except Exception as e:
    print('❌ Not yet. Hint: np.where(camera > T, 255, 0).astype(np.uint8)')

## 3 · Writing the rules by hand: convolution

**Convolution**: slide a small grid of weights (the kernel) over the image; at each position multiply the overlapping numbers and add them up. That sum is one output pixel. *Multiply, add, slide.*

First, the exact example from slide 1.41: an image whose rows are `0 0 0 9 9` and the kernel `-1 0 1`.

In [ ]:
img = np.array([[0, 0, 0, 9, 9]] * 5, dtype=np.float32)
kernel = np.array([[-1, 0, 1]] * 3, dtype=np.float32)

def conv2d(img, kernel):
    """Convolution by hand, no padding, stride 1 (the deep-learning kind: no kernel flip)."""
    kh, kw = kernel.shape
    H, W = img.shape
    out = np.zeros((H - kh + 1, W - kw + 1), dtype=np.float32)
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i+kh, j:j+kw] * kernel)
    return out

result = conv2d(img, kernel)
print(result)

In [ ]:
# Check
try:
    assert np.allclose(result, [[0, 27, 27]] * 3)
    print('✅ Matches the slide: every output row is 0 27 27.')
except Exception as e:
    print('❌ Not yet. Hint: out[i, j] = np.sum(img[i:i+kh, j:j+kw] * kernel)')

### 3.1 · Your convolution vs OpenCV

`cv2.filter2D` does the same multiply-add-slide (it pads the border, so we compare the inside only). Your loop is slow in Python; OpenCV is fast C++ — same maths.

In [ ]:
import time
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
cam = camera.astype(np.float32)
t0 = time.time(); mine = conv2d(cam, sobel_x); t_mine = time.time() - t0
t0 = time.time(); ocv = cv2.filter2D(cam, -1, sobel_x)[1:-1, 1:-1]; t_ocv = time.time() - t0
print(f"same result: {np.allclose(mine, ocv, atol=1e-3)} | yours {t_mine:.2f}s vs OpenCV {t_ocv*1000:.1f}ms")
show(np.abs(mine), titles=["Sobel-x: vertical edges"], cmap="gray")

### 3.2 · The kernel gallery

Same operation, different numbers, completely different results. A kernel gives a large output where the image looks like the kernel.

In [ ]:
kernels = {
    "box blur": np.ones((3, 3), np.float32) / 9,
    "gaussian": cv2.getGaussianKernel(5, 1.2) @ cv2.getGaussianKernel(5, 1.2).T,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),
    "sobel-x": sobel_x,
    "sobel-y": sobel_x.T,
}
outs = [np.clip(np.abs(cv2.filter2D(cam, -1, k)), 0, 255).astype(np.uint8) for k in kernels.values()]
show(*outs, titles=list(kernels), cmap="gray", size=3)

### 3.3 · Edge strength

Sobel-x finds vertical edges, Sobel-y horizontal ones. The edge strength combines both: √(gx² + gy²).

In [ ]:
gx = cv2.filter2D(cam, -1, sobel_x)
gy = cv2.filter2D(cam, -1, sobel_x.T)
mag = np.sqrt(gx**2 + gy**2)
canny = cv2.Canny(camera, 100, 200)
show(mag if mag is not None else cam, canny, titles=["edge strength", "Canny"], cmap="gray")

In [ ]:
# Check
try:
    assert mag.shape == cam.shape and np.allclose(mag, np.hypot(gx, gy))
    print('✅ Edge strength is correct.')
except Exception as e:
    print('❌ Not yet. Hint: mag = np.sqrt(gx**2 + gy**2)')

### Done with the core lab 🎉

You can now read any image as an array, avoid the three classic gotchas (BGR, `[y, x]`, uint8 overflow) and explain convolution with your own code. Tonight's deliverable: finish this notebook and Lab 1B.

## Extra tasks (optional)

1. **HSV colour masking:** find the red parts of the astronaut photo with one range on Hue.
2. **Histogram equalisation:** make a dark version of `camera` and fix it with `cv2.equalizeHist`.
3. **Morphology:** clean the noisy binary mask with an opening (erosion then dilation).
4. **Resizing:** shrink a mask with nearest vs bilinear interpolation and count how many new values appear.

In [ ]:
# Extra 1 — red pixels via Hue
hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
red = (((h < 10) | (h > 170)) & (s > 100) & (v > 60)).astype(np.uint8) * 255
show(rgb, red, titles=["photo", "red mask"], cmap="gray")

In [ ]:
# Extra 2 — equalisation
dark = (camera * 0.3).astype(np.uint8)
eq = cv2.equalizeHist(dark)
show(dark, eq, titles=["dark", "equalised"], cmap="gray")

In [ ]:
# Extra 3 — opening removes specks
rng = np.random.default_rng(0)
noisy = otsu.copy(); noisy[rng.random(noisy.shape) < 0.03] = 255
opened = cv2.morphologyEx(noisy, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
show(noisy, opened, titles=["noisy mask", "opened"], cmap="gray")

In [ ]:
# Extra 4 — masks must be resized with nearest
mask = np.zeros((100, 100), np.uint8); mask[20:60, 30:80] = 1; mask[60:90, 10:50] = 2
near = cv2.resize(mask, (37, 37), interpolation=cv2.INTER_NEAREST)
bil = cv2.resize(mask.astype(np.float32), (37, 37), interpolation=cv2.INTER_LINEAR)
print("nearest values:", np.unique(near), "| bilinear distinct values:", len(np.unique(bil)))